# 05 — OneLake Shortcuts (zero-copy federation)

## Story beat: "Don't migrate — point at it"

Contoso acquired a regional chain whose data still lives in another lakehouse (or in **ADLS**, **S3**, or **GCS**). Instead of copying terabytes, we create a **Shortcut** — a pointer in OneLake that makes external data appear under `Files/` as if it were local. **No bytes move. No second copy to govern.**

---

## UI vs code

| Method | When to use |
| --- | --- |
| **UI** (Module 1 README §1.4) | **Recommended on stage** — visual, reliable |
| **REST API** (this notebook) | Automation, CI/CD, multi-shortcut provisioning |

This notebook is **defensive** — it reports success/failure without crashing. Self-referential shortcuts may be rejected by the API; that's OK.

> **Presenter note:** *"Post-merger, multi-cloud, ministry data sharing — shortcuts are the headline zero-copy story beyond Delta tables."*

In [ ]:
%run 00_config

## Step 1 — Resolve workspace + lakehouse context

Fabric notebooks expose runtime context (workspace id, default lakehouse id). We need these for the Fabric REST API. Authentication uses the notebook's **user/delegated token** via `notebookutils.credentials`.

In [ ]:
import json
import requests

ctx = notebookutils.runtime.context
ws_id = ctx.get("currentWorkspaceId")
lh_id = ctx.get("defaultLakehouseId")
print("workspace:", ws_id, "| lakehouse:", lh_id)

try:
    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    print("token acquired")
except Exception as e:
    headers = None
    print("could not get token:", e)

## Step 2 — Create a OneLake shortcut (best effort)

Target: this workspace's **gold schema** exposed under `Files/gold_shortcut`. If the API rejects it, create the shortcut manually in the UI — see README §1.4.

In [ ]:
created = False
if headers and ws_id and lh_id:
    body = {
        "path": "Files",
        "name": "gold_shortcut",
        "target": {
            "oneLake": {
                "workspaceId": ws_id,
                "itemId": lh_id,
                "path": f"Tables/{GOLD_SCHEMA}",
            }
        },
    }
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items/{lh_id}/shortcuts"
    try:
        r = requests.post(url, headers=headers, data=json.dumps(body))
        print("create status:", r.status_code)
        print(r.text[:400])
        created = r.status_code in (200, 201)
    except Exception as e:
        print("create call error:", e)
else:
    print("Skipping create — use UI method in README §1.4.")

## Step 3 — Read through the shortcut

If the shortcut exists, Spark reads Delta **through the pointer** — same gold tables, zero copy. Failure here is expected if Step 2 didn't succeed.

In [ ]:
try:
    display(notebookutils.fs.ls("Files/gold_shortcut"))
    df = spark.read.format("delta").load("Files/gold_shortcut/sales_by_category")
    display(df)
except Exception as e:
    print("Shortcut not readable here (use UI — README §1.4).")
    print(e)

---

## Template — shortcut to **external** storage (ADLS / S3 / GCS)

Requires a Fabric **connection** to the external account (create once in portal → pass `connectionId`):

```python
body = {
  "path": "Files", "name": "ext_lake",
  "target": {"adlsGen2": {
     "location": "https://<account>.dfs.core.windows.net",
     "subpath": "/<container>/<folder>",
     "connectionId": "<your-connection-guid>"}}
}
# S3: use "amazonS3": {"location": "https://<bucket>.s3.<region>.amazonaws.com", ...}
```

**Trusted Workspace Access (Module 7):** When storage firewall blocks public access, the workspace identity is allow-listed — shortcuts still work over the private backbone.

---

## ✅ Success looks like

- Shortcut visible under `lh_retail → Files/` (UI or API).
- Can query gold data through the shortcut path.

**Next:** `06_cross_engine_reads` — capstone: one copy, many engines.